# Climate Change Impact Analysis Using Historical Data (2000-2023)
**Course:** Data Analytics Using Python (CSL-0312) | **Program:** B.Tech CSE (AIML), Semester III
**Team Leader:** Nagesh Narwariya (BETN1AI25091) | **Faculty Mentor:** Dr. Sanjay Jain

*Note: This notebook relies purely on standard Data Analytics libraries (Pandas, NumPy, Matplotlib, Seaborn, SciPy). No Machine Learning libraries are used.*

In [1]:
# 1. Import Core Data Analytics Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# 2. Hide Warning Messages
import warnings
warnings.filterwarnings('ignore')

# 3. Basic Chart Styling
sns.set_theme(style='whitegrid')

In [2]:
# Load Dataset and Rename Columns
df = pd.read_csv('climate_change_dataset.csv')
df.columns = ['Year', 'Country', 'Temperature', 'CO2_Emissions', 'Sea_Level', 'Rainfall', 'Population', 'Renewable_Energy', 'Extreme_Events', 'Forest_Area']
print('Initial Raw Shape:', df.shape)

In [3]:
# --- DATA CLEANING & PREPROCESSING ---

# STEP 1: Handle Duplicate Rows
df = df.drop_duplicates().reset_index(drop=True)

# STEP 2: Standardize Country Names & Trim Spaces
df['Country'] = df['Country'].astype(str).str.strip().str.title()
df['Country'] = df['Country'].replace({
    'Usa': 'USA', 'U.S.A.': 'USA', 'United States': 'USA',
    'Uk': 'UK', 'U.K.': 'UK', 'United Kingdom': 'UK',
    'India ': 'India'
})

# STEP 3: Handle Missing Values (Imputation with Median)
num_cols = ['Temperature', 'CO2_Emissions', 'Sea_Level', 'Rainfall', 'Renewable_Energy', 'Forest_Area']
for col in num_cols:
    df[col] = df[col].fillna(df.groupby('Country')[col].transform('median'))
    df[col] = df[col].fillna(df[col].median())

# STEP 4: Remove Impossible / Outlier Values
valid_rain = df['Rainfall'] >= 0
valid_renewable = df['Renewable_Energy'] <= 100
valid_temp = df['Temperature'] < 50
df = df[valid_rain & valid_renewable & valid_temp].reset_index(drop=True)

# STEP 5: Create Period Feature for later tests
df['Period'] = np.where(df['Year'] <= 2011, 'Early (2000-2011)', 'Late (2012-2023)')
print('Final Cleaned Shape:', df.shape)

In [4]:
# --- EXPLORATORY DATA ANALYSIS (EDA) ---
summary = df[num_cols].describe().T
summary['median'] = df[num_cols].median()
summary['iqr'] = df[num_cols].quantile(0.75) - df[num_cols].quantile(0.25)
summary['skewness'] = df[num_cols].skew()
summary['kurtosis'] = df[num_cols].kurt()
print(summary.round(3))

In [5]:
# --- VISUALIZATION: Distributions ---
plot_cols = ['Temperature', 'CO2_Emissions', 'Sea_Level', 'Rainfall', 'Population', 'Renewable_Energy', 'Extreme_Events', 'Forest_Area']
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for i, col in enumerate(plot_cols):
    sns.histplot(df[col], kde=True, ax=axes[i], color='blue')
    axes[i].axvline(df[col].mean(), color='red', linestyle='--', label='Mean')
    axes[i].axvline(df[col].median(), color='black', linestyle=':', label='Median')
    axes[i].set_title(col)
    axes[i].legend()

plt.tight_layout()
plt.show()

In [6]:
# --- VISUALIZATION: Colored Country Averages ---
color_map = {
    'Temperature': 'firebrick',
    'CO2_Emissions': 'dimgray',
    'Sea_Level': 'teal',
    'Rainfall': 'royalblue',
    'Population': 'mediumpurple',
    'Renewable_Energy': 'goldenrod',
    'Extreme_Events': 'crimson',
    'Forest_Area': 'forestgreen'
}

country_means = df.groupby('Country')[list(color_map.keys())].mean()
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for i, col in enumerate(color_map.keys()):
    sorted_series = country_means[col].sort_values(ascending=False)
    sns.barplot(x=sorted_series.values, y=sorted_series.index, ax=axes[i], color=color_map[col])
    axes[i].set_title(col, fontweight='bold')
    axes[i].set_xlabel('')
    axes[i].set_ylabel('')

plt.tight_layout()
plt.show()

In [7]:
# --- STATISTICAL ANALYSIS: ANOVA ---
print('Do countries have significantly different temperatures?')
groups = [country_data['Temperature'] for country, country_data in df.groupby('Country')]
f_stat, p_val = stats.f_oneway(*groups)
print(f'ANOVA P-value for Countries: {p_val:.3f}')